In [1]:
!pip install imbalance-metrics smogn

In [2]:
!pip install ImbalancedLearningRegression


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.2/77.2 kB 3.0 MB/s eta 0:00:00


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import PchipInterpolator
from smogn import phi, phi_ctrl_pts
from sklearn.model_selection import KFold
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
import seaborn as sns
from sklearn.neighbors import KNeighborsRegressor
from scipy.spatial import distance
from scipy.spatial import distance_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor,
    BaggingRegressor
)
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.linear_model import (
    ARDRegression,
    SGDRegressor,
    PassiveAggressiveRegressor
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from smogn import smoter
import ImbalancedLearningRegression as iblr

In [4]:
# models_pool = [
#     SVR(),
#      MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
#      XGBRegressor(n_estimators=300, learning_rate=0.05),
#     RandomForestRegressor(n_estimators=300)
# ]
models_pool = [
    SVR(),
    RandomForestRegressor(n_estimators=100, random_state=42),
    DecisionTreeRegressor(random_state=42),
    MLPRegressor(hidden_layer_sizes=(50,50), max_iter=1000, random_state=42),
    BaggingRegressor(random_state=42),
    XGBRegressor(n_estimators=100, random_state=42)
]
# models_pool = [
#     RandomForestRegressor(n_estimators=100, random_state=42),
#     ExtraTreesRegressor(n_estimators=100, random_state=42),
#     XGBRegressor(n_estimators=100, random_state=42)
# ]

In [5]:
## load dependencies - third party
import numpy as np
import pandas as pd
import random as rd
from tqdm import tqdm


## generate synthetic observations
def over_sampling_ro(

    ## arguments / inputs
    data,       ## training set
    index,      ## index of input data
    perc,       ## over / under sampling
    replace,    ## sampling replacement (bool)

    ):

    """
    generates synthetic observations and is the primary function underlying the
    over-sampling technique utilized in the higher main function 'ro()', the
    4 step procedure for generating synthetic observations is:

    1) pre-processing: temporarily removes features without variation, label
    encodes nominal / categorical features, and subsets the training set into
    two data sets by data type: numeric / continuous, and nominal / categorical

    2) over-sampling: RO, which random choose samples from the original samples

    3) post processing: restores original values for label encoded features,
    reintroduces constant features previously removed, converts any interpolated
    negative values to zero in the case of non-negative features

    returns a pandas dataframe containing synthetic observations of the training
    set which are then returned to the higher main function 'ro()'

    ref:

    Branco, P., Torgo, L., Ribeiro, R. (2017).
    SMOGN: A Pre-Processing Approach for Imbalanced Regression.
    Proceedings of Machine Learning Research, 74:36-50.
    http://proceedings.mlr.press/v74/branco17a/branco17a.pdf.

    Branco, P., Ribeiro, R., Torgo, L. (2017).
    Package 'UBL'. The Comprehensive R Archive Network (CRAN).
    https://cran.r-project.org/web/packages/UBL/UBL.pdf.

    Branco, P., Torgo, L., & Ribeiro, R. P. (2019).
    Pre-processing approaches for imbalanced distributions in regression.
    Neurocomputing, 343, 76-99.
    https://www.sciencedirect.com/science/article/abs/pii/S0925231219301638

    Kunz, N., (2019). SMOGN.
    https://github.com/nickkunz/smogn
    """

    ## subset original dataframe by bump classification index
    data = data.iloc[index]

    ## store dimensions of data subset
    n = len(data)
    d = len(data.columns)

    ## store original data types
    feat_dtypes_orig = [None] * d

    for j in range(d):
        feat_dtypes_orig[j] = data.iloc[:, j].dtype

    ## find non-negative numeric features
    feat_non_neg = []
    num_dtypes = ["int64", "float64"]

    for j in range(d):
        if data.iloc[:, j].dtype in num_dtypes and any(data.iloc[:, j] > 0):
            feat_non_neg.append(j)

    ## find features without variation (constant features)
    feat_const = data.columns[data.nunique() == 1]

    ## temporarily remove constant features
    if len(feat_const) > 0:

        ## create copy of orignal data and omit constant features
        data_orig = data.copy()
        data = data.drop(data.columns[feat_const], axis = 1)

        ## store list of features with variation
        feat_var = list(data.columns.values)

        ## reindex features with variation
        for i in range(d - len(feat_const)):
            data.rename(columns = {
                data.columns[i]: i
                }, inplace = True)

        ## store new dimension of feature space
        d = len(data.columns)

    ## create copy of data containing variation
    data_var = data.copy()

    ## create global feature list by column index
    feat_list = list(data.columns.values)

    ## create nominal feature list and
    ## label encode nominal / categorical features
    ## (strictly label encode, not one hot encode)
    feat_list_nom = []
    nom_dtypes = ["object", "bool", "datetime64"]

    # Unknown warning, may be handled later
    pd.options.mode.chained_assignment = None

    for j in range(d):
        if data.dtypes[j] in nom_dtypes:
            feat_list_nom.append(j)
            data.iloc[:, j] = pd.Categorical(pd.factorize(
                data.iloc[:, j])[0])

    data = data.apply(pd.to_numeric)

    ## create numeric feature list
    feat_list_num = list(set(feat_list) - set(feat_list_nom))

    ## calculate ranges for numeric / continuous features
    ## (includes label encoded features)
    feat_ranges = list(np.repeat(1, d))

    if len(feat_list_nom) > 0:
        for j in feat_list_num:
            feat_ranges[j] = max(data.iloc[:, j]) - min(data.iloc[:, j])
    else:
        for j in range(d):
            feat_ranges[j] = max(data.iloc[:, j]) - min(data.iloc[:, j])

    ## subset feature ranges to include only numeric features
    ## (excludes label encoded features)
    feat_ranges_num = [feat_ranges[i] for i in feat_list_num]

    ## subset data by either numeric / continuous or nominal / categorical
    data_num = data.iloc[:, feat_list_num]
    data_nom = data.iloc[:, feat_list_nom]

    ## get number of features for each data type
    feat_count_num = len(feat_list_num)
    feat_count_nom = len(feat_list_nom)


    ## number of new synthetic observations for each rare observation
    x_synth = int(perc - 1)

    ## total number of new synthetic observations to generate
    n_synth = int(n * (perc - 1 - x_synth))

    ## randomly index data by the number of new synthetic observations
    r_index = np.random.choice(
        a = tuple(range(0, n)),
        size = x_synth * n + n_synth if replace else n_synth,
        replace = replace,
        p = None
    )

    ## create null matrix to store new synthetic observations
    synth_matrix = np.ndarray(shape = ((x_synth * n + n_synth), d))

    # added
    ## store data in the synthetic matrix, data indices are chosen randomly above
    count = 0
    for i in tqdm(r_index, ascii = True, desc = "r_index"):
        for attr in range(d):
            synth_matrix[count, attr] = (data.iloc[i, attr])
        count = count + 1
    ## if the number of random chosen samples is greater than the number of samples，
    ## and replace = False,
    ## simply duplicate x_synth times the original samples
    if not replace:
        for i in tqdm(range(x_synth * n), ascii = True, desc = "duplicating_the_same_samples"):
            for attr in range(d):
                synth_matrix[count, attr] = (data.iloc[i % n, attr])
            count = count + 1


    ## convert synthetic matrix to dataframe
    data_new = pd.DataFrame(synth_matrix)

    ## synthetic data quality check
    if sum(data_new.isnull().sum()) > 0:
        raise ValueError("oops! synthetic data contains missing values")

    ## replace label encoded values with original values
    for j in feat_list_nom:
        code_list = data.iloc[:, j].unique()
        cat_list = data_var.iloc[:, j].unique()

        for x in code_list:
            data_new.iloc[:, j] = data_new.iloc[:, j].replace(x, cat_list[x])

    ## reintroduce constant features previously removed
    if len(feat_const) > 0:
        data_new.columns = feat_var

        for j in range(len(feat_const)):
            data_new.insert(
                loc = int(feat_const[j]),
                column = feat_const[j],
                value = np.repeat(
                    data_orig.iloc[0, feat_const[j]],
                    len(synth_matrix))
            )

    ## convert negative values to zero in non-negative features
    for j in feat_non_neg:
        # data_new.iloc[:, j][data_new.iloc[:, j] < 0] = 0
        data_new.iloc[:, j] = data_new.iloc[:, j].clip(lower = 0)

    ## return over-sampling results dataframe
    return data_new

In [6]:
import numpy as np

def distance(y, D=1.0, delta=1.0):
    """
    Distance-based relevance function φ_D
    y: array-like (assumido padronizado)
    """
    y = np.asarray(y)
    n = len(y)
    raw_scores = np.zeros(n)

    for j in range(n):
        s = 0.0
        for i in range(n):
            d = abs(y[j] - y[i])
            if 0 < d < D:
                s += 1.0 / (d + delta)
        raw_scores[j] = s

    s_min = raw_scores.min()
    s_max = raw_scores.max()

    if s_max == s_min:
        raise ValueError("redefine relevance function: all points have same density")

    dist = (s_max - raw_scores) / (s_max - s_min)
    return dist


In [7]:
def apply_ro_dist(
    data,
    y,
    samp_method="balance",
    drop_na_col=True,
    drop_na_row=True,
    replace=True,
    manual_perc=False,
    perc_o=-1,
    rel_thres=0.5,
    rel_method="auto",
    rel_xtrm_type="both",
    rel_coef=1.5,
    rel_ctrl_pts_rg=None
):
    """
    Versão do ro() mantendo todo o fluxo original, mas substituindo a etapa
    de phi(y) por Instance Hardness (IH) calculado pela função
    instance_hardness_regression. Basta passar `ih_models` (lista de modelos).
    """

    ## PRE-PROCESSAMENTO (IGUAL AO ORIGINAL)
    if bool(drop_na_col):
        data = data.dropna(axis=1)

    if bool(drop_na_row):
        data = data.dropna(axis=0)

    if data.isnull().values.any():
        raise ValueError("cannot proceed: data cannot contain NaN values")

    if not isinstance(y, str):
        raise ValueError("y must be a string")

    if y not in data.columns:
        raise ValueError("y must be a header in dataframe")

    if samp_method not in ["balance", "extreme"]:
        raise ValueError("samp_method must be 'balance' or 'extreme'")

    if manual_perc:
        if perc_o == -1:
            raise ValueError("require percentage of over-sampling")
        if perc_o <= 0:
            raise ValueError("percentage must be positive real")

    if rel_thres <= 0 or rel_thres > 1:
        raise ValueError("rel_thres must be 0 < R < 1")
    n = len(data)
    d = len(data.columns)

    feat_dtypes_orig = [data.iloc[:, j].dtype for j in range(d)]

    # SALVAR cópia do índice original (usado para alinhar IH)
    original_index = data.index.copy()

    # -------------------------
    # mover y para última coluna
    # -------------------------
    y_col = data.columns.get_loc(y)
    if y_col < d - 1:
        cols = list(range(d))
        cols[y_col], cols[d - 1] = cols[d - 1], cols[y_col]
        data = data[data.columns[cols]]

    feat_names = list(data.columns)
    data.columns = range(d)

    # -------------------------
    # ordenar Y
    # -------------------------
    y_df = pd.DataFrame(data[d - 1])
    y_sort = y_df.sort_values(by=d - 1)
    y_sort_series = y_sort[d - 1]

    # =========================================================
    # 🔁 SUBSTITUIÇÃO DO IH POR φᴰ (APENAS Y)
    # =========================================================

    y_sorted_values = y_sort_series.values

    # padronização
    y_std = (y_sorted_values - np.mean(y_sorted_values)) / np.std(y_sorted_values)

    # cálculo da relevância φᴰ
    dist_y = distance(
        y=y_std,
        D=1.0,
        delta=1.0
    )

    if np.all(dist_y == 0):
        raise ValueError("redefine relevance function: all points are 0")

    if np.all(dist_y == 1):
        raise ValueError("redefine relevance function: all points are 1")

    # -------------------------
    # determinar bumps (φᴰ > 0.5)
    # -------------------------
    bumps = [0]
    rel_thres = 0.5

    for i in range(len(dist_y) - 1):
        if ((dist_y[i] >= rel_thres and dist_y[i + 1] < rel_thres) or
            (dist_y[i] < rel_thres and dist_y[i + 1] >= rel_thres)):
            bumps.append(i + 1)

    bumps.append(n)
    n_bumps = len(bumps) - 1

    b_index = {}
    for i in range(n_bumps):
        # aqui usamos y_sort (Series) com slicing para preservar índices originais,
        # como no seu código original.
        b_index.update({i: y_sort_series[bumps[i]:bumps[i + 1]]})

    # cálculo de s_perc idêntico ao original
    b = round(n / n_bumps)
    s_perc = []
    scale = []
    obj = []

    if samp_method == "balance":
        for i in b_index:
            s_perc.append(b / len(b_index[i]))

    if samp_method == "extreme":
        for i in b_index:
            scale.append(b ** 2 / len(b_index[i]))
        scale = n_bumps * b / sum(scale)

        for i in b_index:
            obj.append(round(b ** 2 / len(b_index[i]) * scale, 2))
            s_perc.append(round(obj[i] / len(b_index[i]), 1))

    data_new = pd.DataFrame()

    for i in range(n_bumps):

        if s_perc[i] == 1:
            # b_index[i] é uma Series (y values) com índices originais
            data_new = pd.concat(
                [data.iloc[b_index[i].index], data_new], ignore_index=True
            )

        if s_perc[i] > 1:

            synth_obs = over_sampling_ro(
                data=data,
                index=list(b_index[i].index),
                perc=s_perc[i] if not manual_perc else perc_o + 1,
                replace=replace
            )

            data_new = pd.concat([synth_obs, data_new], ignore_index=True)

            original_obs = data.iloc[list(b_index[i].index)]
            data_new = pd.concat([original_obs, data_new], ignore_index=True)

        if s_perc[i] < 1:
            original_obs = data.iloc[list(b_index[i].index)]
            data_new = pd.concat([original_obs, data_new], ignore_index=True)

    # restaurar nomes das colunas originais
    data_new.columns = feat_names

    # restaurar posição original de y, se necessário
    if y_col < d - 1:
        cols = list(range(d))
        cols[y_col], cols[d - 1] = cols[d - 1], cols[y_col]
        data_new = data_new[data_new.columns[cols]]

    # restaurar tipos originais
    for j in range(d):
        data_new.iloc[:, j] = data_new.iloc[:, j].astype(feat_dtypes_orig[j])

    return data_new

In [8]:

def evaluate_model(X_train, X_test, y_train, y_test, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # MSE
    mse = mean_squared_error(y_test, y_pred)

    # mae
    mae = mean_absolute_error(y_test, y_pred)

    return mse, mae


In [9]:
models = {
    "SVR": SVR(),  # não possui random_state
    "NNET":MLPRegressor(hidden_layer_sizes=(50,50), max_iter=1000, random_state=42),
    "XGB":XGBRegressor(n_estimators=100, random_state=42),
    "RF":  RandomForestRegressor(n_estimators=100, random_state=42),
    "BAGGING": BaggingRegressor(random_state=42)
}

In [10]:
wine = pd.read_csv('/content/drive/MyDrive/TCC/dados/wine_quality.csv')
a1 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a1.csv')
a2 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a2.csv')
a3 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a3.csv')
a7 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a7.csv')
acceleration = pd.read_csv('/content/drive/MyDrive/TCC/dados/acceleration.csv')
airfoild = pd.read_csv('/content/drive/MyDrive/TCC/dados/airfoild.csv')
boston = pd.read_csv('/content/drive/MyDrive/TCC/dados/boston.csv')
concreteStrength = pd.read_csv('/content/drive/MyDrive/TCC/dados/concreteStrength.csv')
heat = pd.read_csv('/content/drive/MyDrive/TCC/dados/heat.csv')
sensory = pd.read_csv('/content/drive/MyDrive/TCC/dados/sensory.csv')
analcatdata_apnea3 = pd.read_csv('/content/drive/MyDrive/TCC/dados/analcatdata_apnea3.csv')
available_power = pd.read_csv('/content/drive/MyDrive/TCC/dados/available_power.csv')
cocomo_numeric = pd.read_csv('/content/drive/MyDrive/TCC/dados/cocomo_numeric.csv')
debutanizer = pd.read_csv('/content/drive/MyDrive/TCC/dados/debutanizer.csv')
fuel_consumption_country = pd.read_csv('/content/drive/MyDrive/TCC/dados/fuel_consumption_country.csv')

abalone = pd.read_csv('/content/drive/MyDrive/TCC/dados/abalone.csv')
california = pd.read_csv('/content/drive/MyDrive/TCC/dados/california.csv')
compactiv = pd.read_csv('/content/drive/MyDrive/TCC/dados/compactiv.csv')
cpu_small = pd.read_csv('/content/drive/MyDrive/TCC/dados/cpu_small.csv')
forestFires = pd.read_csv('/content/drive/MyDrive/TCC/dados/forestFires.csv')
kdd_coil_1 = pd.read_csv('/content/drive/MyDrive/TCC/dados/kdd_coil_1.csv')
lungcancer_shedden = pd.read_csv('/content/drive/MyDrive/TCC/dados/lungcancer_shedden.csv')
maximal_torque = pd.read_csv('/content/drive/MyDrive/TCC/dados/maximal_torque.csv')
meta = pd.read_csv('/content/drive/MyDrive/TCC/dados/meta.csv')
mortgage = pd.read_csv('/content/drive/MyDrive/TCC/dados/mortgage.csv')
pdgfr = pd.read_csv('/content/drive/MyDrive/TCC/dados/pdgfr.csv')
space_ga = pd.read_csv('/content/drive/MyDrive/TCC/dados/space_ga.csv')
treasury = pd.read_csv('/content/drive/MyDrive/TCC/dados/treasury.csv')
triazines = pd.read_csv('/content/drive/MyDrive/TCC/dados/triazines.csv')



datasets = {
    "a1": {"data": a1, "target": "a1"},
    "a2": {"data": a2, "target": "a2"},
    "a3": {"data": a3, "target": "a3"},
    "a7": {"data": a7, "target": "a7"},
     "abalone": {"data": abalone, "target": "Rings"},
     "acceleration": {"data": acceleration, "target": "target"},
    "airfoild": {"data": airfoild, "target": "target"},
    "analcatdata_apnea3": {"data": analcatdata_apnea3, "target": "target"},
     "available_power": {"data": available_power, "target": "target"},
   "boston": {"data": boston, "target": "target"},
    #"california": {"data": california, "target": "target"},
    "cocomo_numeric": {"data": cocomo_numeric, "target": "target"},
    "compactiv": {"data": compactiv, "target": "target"},
     "concreteStrength": {"data": concreteStrength, "target": "target"},
     "cpu_small": {"data": cpu_small, "target": "usr"},
     "debutanizer": {"data": debutanizer, "target": "target"},
    "fuel_consumption_country": {"data": fuel_consumption_country, "target": "fuel.consumption.country"},
    "forestFires": {"data": forestFires, "target": "Area"},
     "heat": {"data": heat, "target": "heat"},
    "kdd_coil_1": {"data": kdd_coil_1, "target": "target"},
    "lungcancer_shedden": {"data": lungcancer_shedden, "target": "target"},
     "maximal_torque": {"data": maximal_torque, "target": "maximal.torque"},
    "meta": {"data": meta, "target": "target"},
    "mortgage": {"data": mortgage, "target": "target"},
    "pdgfr": {"data": pdgfr, "target": "target"},
    "sensory": {"data": sensory, "target": "target"},
     "space_ga": {"data": space_ga, "target": "target"},
    "treasury": {"data": treasury, "target": "target"},
    "triazines": {"data": triazines, "target": "target"},
    "wine": {"data": wine, "target": "target"},
}


Teste para RO

In [11]:
from sklearn.model_selection import RepeatedKFold

rkf = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

results = []

for name, info in datasets.items():
    df = info["data"].copy()
    target = info["target"]

    print(f"\nProcessando: {name}")

    X = df.drop(columns=[target])
    y = df[target]

    # Loop RepeatedKFold
    for fold_id, (train_idx, test_idx) in enumerate(rkf.split(X)):
        repeat = fold_id // 5 + 1
        fold = fold_id % 5 + 1

        print(f"  Repetição {repeat}/2 - Fold {fold}/5")

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        base_train = pd.concat([X_train, y_train], axis=1).reset_index(drop=True)

        # Oversampling dentro do fold
        try:
            train_hard = apply_ro_dist(base_train, target)
        except Exception as e:
            print(f"⚠️ RO falhou no fold {fold} da repetição {repeat}: {e}")
            train_hard = base_train.copy()

        versions = {
            "Distance-RO": train_hard
        }

        for model_name, model in models.items():
            for version_name, train_data in versions.items():

                Xtr = train_data.drop(columns=[target])
                ytr = train_data[target]

                Xts = X_test
                yts = y_test

                # Ajuste e predição
                model.fit(Xtr, ytr)
                y_pred = model.predict(Xts)

                # Cálculo das métricas
                mse = mean_squared_error(yts, y_pred)
                mae = mean_absolute_error(yts, y_pred)

                results.append({
                    "Dataset": name,
                    "Repeticao": repeat,
                    "Fold": fold,
                    "Model": model_name,
                    "Versao": version_name,
                    "MSE": mse,
                    "MAE": mae,
                    "y_pred": y_pred.tolist(),   # <<<<<<<< AQUI EXPORTA O Y PREDITO
                    "y_true": yts.tolist()        # (opcional mas recomendado)
                })



Processando: a1
  Repetição 1/2 - Fold 1/5


r_index: 100%|##########| 38/38 [00:00<00:00, 4776.25it/s]


  Repetição 1/2 - Fold 2/5


r_index: 100%|##########| 37/37 [00:00<00:00, 5186.81it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


  Repetição 1/2 - Fold 3/5


...

  Repetição 2/2 - Fold 4/5


r_index: 100%|##########| 46/46 [00:00<00:00, 1964.91it/s]


  Repetição 2/2 - Fold 5/5


r_index: 100%|##########| 46/46 [00:00<00:00, 2400.65it/s]



Processando: wine
  Repetição 1/2 - Fold 1/5
⚠️ RO falhou no fold 1 da repetição 1: redefine relevance function: all points have same density
  Repetição 1/2 - Fold 2/5
⚠️ RO falhou no fold 2 da repetição 1: redefine relevance function: all points have same density
  Repetição 1/2 - Fold 3/5
⚠️ RO falhou no fold 3 da repetição 1: redefine relevance function: all points have same density
  Repetição 1/2 - Fold 4/5
⚠️ RO falhou no fold 4 da repetição 1: redefine relevance function: all points have same density
  Repetição 1/2 - Fold 5/5
⚠️ RO falhou no fold 5 da repetição 1: redefine relevance function: all points have same density
  Repetição 2/2 - Fold 1/5
⚠️ RO falhou no fold 1 da repetição 2: redefine relevance function: all points have same density
  Repetição 2/2 - Fold 2/5
⚠️ RO falhou no fold 2 da repetição 2: redefine relevance function: all points have same density
  Repetição 2/2 - Fold 3/5
⚠️ RO falhou no fold 3 da repetição 2: redefine relevance function: all points have sa

In [12]:
results_df = pd.DataFrame(results)
results_df.head(50)


,Dataset,Repeticao,Fold,Model,Versao,MSE,MAE,y_pred,y_true
0,a1,1,1,SVR,Distance-RO,3.874273e+02,14.951894,"[23.104418264174633, 23.10412537285756, 16.683...","[17.0, 69.9, 46.2, 50.6, 0.0, 13.0, 3.4, 29.5,..."
1,a1,1,1,NNET,Distance-RO,9.359439e+05,312.366760,"[13.280284767152915, 14.533050666756766, 99.48...","[17.0, 69.9, 46.2, 50.6, 0.0, 13.0, 3.4, 29.5,..."
2,a1,1,1,XGB,Distance-RO,3.981851e+02,14.726173,"[17.565092086791992, 53.407997131347656, 32.32...","[17.0, 69.9, 46.2, 50.6, 0.0, 13.0, 3.4, 29.5,..."
3,a1,1,1,RF,Distance-RO,3.059693e+02,14.314600,"[42.268, 46.42800000000004, 32.790000000000006...","[17.0, 69.9, 46.2, 50.6, 0.0, 13.0, 3.4, 29.5,..."
4,a1,1,1,BAGGING,Distance-RO,3.048272e+02,13.930750,"[44.56000000000001, 51.55, 25.979999999999997,...","[17.0, 69.9, 46.2, 50.6, 0.0, 13.0, 3.4, 29.5,..."
5,a1,1,2,SVR,Distance-RO,3.664788e+02,14.646652,"[7.626668725504199, 16.486543911120215, 19.098...","[3.3, 15.1, 43.5, 29.7, 33.9, 6.9, 18.3, 66.0,..."
6,a1,1,2,NNET,Distance-RO,1.762711e+05,185.308923,"[76.06398364811676, 799.0115343878548, 39.7304...","[3.3, 15.1, 43.5, 29.7, 33.9, 6.9, 18.3, 66.0,..."
7,a1,1,2,XGB,Distance-RO,2.884696e+02,11.054897,"[3.519385814666748, 14.375164985656738, 44.743...","[3.3, 15.1, 43.5, 29.7, 33.9, 6.9, 18.3, 66.0,..."
8,a1,1,2,RF,Distance-RO,2.037181e+02,10.357400,"[8.066, 22.380999999999993, 39.89800000000002,...","[3.3, 15.1, 43.5, 29.7, 33.9, 6.9, 18.3, 66.0,..."
9,a1,1,2,BAGGING,Distance-RO,2.639873e+02,11.492500,"[6.530000000000001, 18.94, 43.59000000000001, ...","[3.3, 15.1, 43.5, 29.7, 33.9, 6.9, 18.3, 66.0,..."


In [13]:
results_df.to_csv("Distance_RO.csv", index=False)